# ISLES 2022: first modern nnU-Net v2 baseline (DWI + ADC)

This notebook is the first executable **modernization baseline**. It does not replace the historical ISLES 2015 reproduction.

Why DWI + ADC first? They are the core acute-stroke sequences, and FLAIR is intentionally held out until the real geometry audit tells us whether explicit registration is required. The staging code will also refuse DWI/ADC if their geometry does not match.

The notebook prepares a reproducible nnU-Net v2.8.1 experiment and provides training commands for all five folds. A full 5-fold training campaign can take many GPU-hours, so the expensive cells are deliberately separated from data validation/preprocessing.


In [ ]:
!nvidia-smi || true
!rm -rf HybridStrokeSeg-Reproduction
!git clone -q https://github.com/NTinkicht/HybridStrokeSeg-Reproduction.git
%cd HybridStrokeSeg-Reproduction
!python -m pip install -q -e '.[notebook,modern]'
!python -c "import importlib.metadata as m; print('nnunetv2', m.version('nnunetv2'))"


## 1. Download and verify ISLES 2022


In [ ]:
!python scripts/download_isles2022.py --extract


## 2. Audit geometry before staging
No registration or resampling is silently performed.


In [ ]:
!python scripts/audit_isles2022.py data/raw/isles2022 --json outputs/isles2022_geometry.json
import json
from pathlib import Path
audit = json.loads(Path('outputs/isles2022_geometry.json').read_text())
audit


## 3. Define isolated nnU-Net paths
The raw, preprocessed and model-result trees are kept separate from the source dataset.


In [ ]:
import os
from pathlib import Path

workspace = Path('/content/nnunet_workspace')
nn_raw = workspace / 'nnUNet_raw'
nn_preprocessed = workspace / 'nnUNet_preprocessed'
nn_results = workspace / 'nnUNet_results'
for folder in (nn_raw, nn_preprocessed, nn_results):
    folder.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(nn_raw)
os.environ['nnUNet_preprocessed'] = str(nn_preprocessed)
os.environ['nnUNet_results'] = str(nn_results)
print('nnUNet_raw =', nn_raw)
print('nnUNet_preprocessed =', nn_preprocessed)
print('nnUNet_results =', nn_results)


## 4. Stage DWI + ADC with a hard geometry gate
If this cell stops with a geometry error, that is a useful result: we must define and validate registration before training rather than hiding it inside preprocessing.


In [ ]:
!python scripts/stage_isles2022_nnunet.py data/raw/isles2022 {nn_raw} --dataset-id 501 --dataset-name ISLES2022_DWI_ADC --channels dwi adc --mode symlink --overwrite
dataset_raw = nn_raw / 'Dataset501_ISLES2022_DWI_ADC'
print((dataset_raw / 'dataset.json').read_text())


## 5. nnU-Net fingerprinting, planning and preprocessing
Dataset integrity checking is enabled on the first run, as recommended by nnU-Net.


In [ ]:
!nnUNetv2_plan_and_preprocess -d 501 --verify_dataset_integrity -c 3d_fullres


## 6. Force the repository's deterministic patient-level folds


In [ ]:
import shutil
preprocessed_dataset = nn_preprocessed / 'Dataset501_ISLES2022_DWI_ADC'
shutil.copy2(dataset_raw / 'splits_final.json', preprocessed_dataset / 'splits_final.json')
folds = json.loads((preprocessed_dataset / 'splits_final.json').read_text())
[(len(f['train']), len(f['val'])) for f in folds]


## 7. Training
For a quick infrastructure check, run only fold 0 first. For reportable cross-validation, train **all five folds with `--npz`**. The `--npz` flag is required for nnU-Net's later best-configuration selection.


In [ ]:
# QUICK GPU CHECK: uncomment to train fold 0.
# !nnUNetv2_train 501 3d_fullres 0 --npz


In [ ]:
# FULL 5-FOLD BASELINE: expensive; run one fold per GPU session/job.
for fold in range(5):
    print(f'nnUNetv2_train 501 3d_fullres {fold} --npz')


## 8. Model selection and inference after all folds finish
nnU-Net's model-selection command writes inference instructions. By default, prediction with a trained configuration uses all five folds as an ensemble.


In [ ]:
# Run only after folds 0-4 have completed:
# !nnUNetv2_find_best_configuration 501 -c 3d_fullres
# print((nn_results / 'Dataset501_ISLES2022_DWI_ADC' / 'inference_instructions.txt').read_text())


## Scientific checkpoint
Do not report this baseline until all five folds are trained and evaluated with the repository's ISLES 2022 metrics. Do not add FLAIR to this model simply to increase the number of channels; first inspect the geometry audit and implement an explicit, validated registration path if needed.
